# Modelos de Regresión Regularizados

El objetivo es predecir el precio de una casa dados sus atributos, utilizando modelos regularizados.

##### Bibliotecas necesarias

In [ ]:
#General
import numpy as np
import pandas as pd
from skimpy import skim

#Estadisticas
from scipy import stats

#Graficas
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pandas.plotting import andrews_curves

#Valores para el CV
from scipy.stats import uniform

#Valores atipicos
from pyod.models.mcd import MCD

#Modelo de regresion 
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression

#Escalamiento
from sklearn.preprocessing import MinMaxScaler

#Busqueda de hiper-parametros, train test y CV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split

#Performance
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

#### Global Options

In [ ]:
np.random.seed(1989)

In [ ]:
pd.options.display.float_format = '{:,.3f}'.format

## Carga de datos

El conjunto de datos consta de las siguientes variables:

* Superficie: La superficie total de la casa en pies cuadrados.
* Dormitorios: El número de dormitorios de la casa.
* Baños: El número de baños de la casa.
* Plantas: El número de plantas de la casa.
* Calle principal: Si la casa está conectada a la calle principal (Sí/No).
* Habitación: Si la casa tiene habitación (Sí/No).
* Sótano: Si la casa tiene sótano (Sí/No).
* Calentador de agua: Si la casa tiene calentador de agua (Sí/No).
* Aire acondicionado: Si la casa tiene aire acondicionado (Sí/No).
* Aparcamiento: El número de plazas de aparcamiento disponibles dentro de la casa.
* Prefacio: Si la casa está ubicada en una zona preferida (Sí/No).
* Estado del mobiliario: El estado del mobiliario de la casa (Totalmente amueblado, Semiamueblado, Sin amueblar).

La variable objetivo es: **Precio** (en dólares).

In [ ]:
#Cargamos los datos
df = pd.read_csv(r"C:\Users\raulb\Desktop\Diplomado\Modulo2\Datos\Housing.csv")

In [ ]:
df.sample(15)

In [ ]:
#Definimos las variables numericas y categoricas
var_num = ["price", "area", "bedrooms", "bathrooms", "stories", "parking"]
var_cat = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"]

In [ ]:
#Transformamos mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea y furnishingstatus
for var in var_cat:
    df[var] = df[var].astype("category")

In [ ]:
#Resumen de los datos
skim(df)

No hay valores ausentes.

## EDA

### Gráficos y correlación

In [ ]:
#Definimos un objeto para graficar
class analyze:
    # Inicializamos la clase
    def __init__(self, df, num_vars, cat_vars):
        self.df = df
        self.num_vars = num_vars
        self.cat_vars = cat_vars

    #Método para graficar variables numéricas
    def plot_num(self):
        for var in self.num_vars:
            fig = make_subplots(rows=1, cols=2)

            # Boxplot
            fig.add_trace(
                go.Box(y=self.df[var], name="Boxplot", marker_color="#de3163"),
                row=1, col=1
            )

            # Histograma
            fig.add_trace(
                go.Histogram(x=self.df[var], name="Histograma",
                             marker_color="#31deac", nbinsx=50,
                             histnorm='probability density'),
                row=1, col=2
            )

            fig.update_layout(title_text=f"Variable: {var}", height=400, width=800)
            fig.show()

    #Método para graficar variables categóricas
    def plot_cat(self):
        for var in self.cat_vars:
            value_counts = self.df[var].value_counts()
            labels = value_counts.index.astype(str)
            values = value_counts.values

            fig = go.Figure(
                data=[go.Pie(labels=labels, values=values,textinfo='label+value+percent',
                    marker=dict(colors=["#de3163", "#31deac"])
                                )
                    ]
            )

            fig.update_layout(title_text=f"Variable: {var}", height=400, width=500)
            fig.show()

    #Método para graficar la matriz de correlación de numéricas
    def corr_matrix(self):
        corr_matrix = self.df[self.num_vars].corr()

        fig = go.Figure(data=go.Heatmap(
            z=corr_matrix.values,
            x=corr_matrix.columns,
            y=corr_matrix.columns,
            colorscale='Inferno'))

        fig.update_layout(
            title='Matriz de Correlación',
            width=800,
            height=600
        )

        fig.show()

In [ ]:
#Definimos la Curva de Andrews escalada
def andrews_curve_scaled(df):
    df_aux = df.copy()
    
    #Escalamos solo las columnas numéricas
    numeric_cols = df_aux.select_dtypes(include=['float64', 'int64']).columns
    scaler = MinMaxScaler()
    df_aux[numeric_cols] = scaler.fit_transform(df_aux[numeric_cols])
    
    #Agregamos una categoría ficticia
    df_aux["dummy_class"] = ""
    
    plt.figure(figsize=(12,8))
    andrews_curves(df_aux, "dummy_class", colormap="viridis", alpha=0.2)
    plt.title("Curvas de Andrews")
    plt.xlabel("t (función de Fourier)")
    plt.ylabel("Valor transformado")
    plt.grid(True, color="lightgray", linestyle="--", linewidth=0.5)
    plt.show()

In [ ]:
analisis = analyze(df, var_num, var_cat)

In [ ]:
analisis.plot_num()

In [ ]:
analisis.plot_cat()

In [ ]:
analisis.corr_matrix()

De lo anterior podemos describir lo siguiente:
* El mínimo de precio es de 1 millón 750 mil dólares; el máximo es de 13 millones 300 mil. Se tiene una desviación estándar de 1 millón 870 mil dólares. 
  En consecuencia, el precio tiene una variabilidad alta y sesgada hacia valores pequeños. Existen más casas con precios bajos o medios que altos. 

* El área también tiene una variabilidad alta. Hay más casas con áreas pequeñas/medias que grandes. 

* El número de habitaciones va de 1 a 6. 

* El número de baños va de 1 a 4. 

* El número de plantas va de 1 a 4. 

* Los lugares de estacionamiento van de 0 a 3. 

* Eliminaremos hotwater, pues hay una dominancia mayor al 90%.

* No hay problemas de multicolinealidad.

* La mayoría de casas tienen tres habitaciones, un baño, dos plantas y cero lugares de estacionamiento.

#### Valores Atípicos

In [ ]:
andrews_curve_scaled(df[var_num])

Podemos ver la existencia de valores atípicos, en particular algunos que sobresalen de las demás curvas. 

In [ ]:
#Usaremos minima covarianza
def outliers_mcd(df, contamination=0.01, random_state=1989, support_fraction=None):
    model = MCD(contamination=contamination, random_state=random_state, support_fraction=support_fraction)
    model.fit(df)
    preds = model.predict(df)
    return df[preds == 1]  #detectar

In [ ]:
df_outliers = outliers_mcd(df[var_num])

In [ ]:
#Visualizamos en el df original
df.loc[df_outliers.index]

In [ ]:
#Eliminamos los atipicos
df.drop(df_outliers.index, inplace=True)

In [ ]:
df

In [ ]:
andrews_curve_scaled(df[var_num])

## Modelado

### Partición, codificación y escalamiento

In [ ]:
#Variables explicativas y variable objetivo
y = df["price"]
X = df.drop(columns=["price", "hotwaterheating"])

In [ ]:
#Codificacion 
X = pd.get_dummies(X, columns=["mainroad", "basement", "guestroom", "furnishingstatus", "prefarea", "airconditioning"], drop_first=True)

In [ ]:
#Particion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1989)

In [ ]:
#Escalamiento
var_num = ["area", "bedrooms", "bathrooms", "stories", "parking"]
scaler = MinMaxScaler()

#Ajustamos solo con train
scaler.fit(X_train)

#Transformar train y test
X_train[var_num] = scaler.fit_transform(X_train[var_num])
X_test[var_num] = scaler.transform(X_test[var_num])

### Regresiones

#### Regresion Ordinaria

In [ ]:
class Regresion:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        
        self.model = None
        self.y_fitted = None
        self.y_pred = None

    #Ajustamos el modelo
    def fit_reg(self):
        self.model = LinearRegression(fit_intercept=True)
        self.model.fit(self.X_train, self.y_train)
        
        #Calculamos predicciones
        self.y_fitted = self.model.predict(self.X_train)
        self.y_pred = self.model.predict(self.X_test)
        return self.model

    #Coeficientes del modelo
    def coefs(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
            
        cfs = [self.model.intercept_]
        cfs.extend(list(self.model.coef_))
        coef_names = ["intercept"] + list(self.X_train.columns)

        dt = {"Vars": coef_names, "Coefs": cfs}
        df_coefs = pd.DataFrame(dt)
        
        return df_coefs

    #Validacion cruzada
    def cross_values(self, cv=4):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
        
        cv_score = cross_val_score(
            estimator=self.model,
            X=self.X_train,
            y=self.y_train,
            cv=cv,
            n_jobs=-1,
            scoring="r2"
        )
        cv_values = {
            "media": np.round(np.mean(cv_score),3),
            "mediana": np.round(np.median(cv_score),3),
            "desv. est": np.round(np.std(cv_score, ddof=1),3)
        }
        return cv_values

    #Metricas de desempeño
    def performance(self):
        if self.y_fitted is None or self.y_pred is None:
            raise ValueError("Primero ajusta el modelo con fit_reg() para calcular predicciones")
        
        r2_train = np.round(r2_score(self.y_train, self.y_fitted), 3)
        r2_test  = np.round(r2_score(self.y_test, self.y_pred), 3)

        rmse_train = np.round(root_mean_squared_error(self.y_train, self.y_fitted), 3)
        rmse_test  = np.round(root_mean_squared_error(self.y_test, self.y_pred), 3)

        mae_train = np.round(mean_absolute_error(self.y_train, self.y_fitted), 3)
        mae_test  = np.round(mean_absolute_error(self.y_test, self.y_pred), 3)

        mape_train = np.round(mean_absolute_percentage_error(self.y_train, self.y_fitted), 3)
        mape_test  = np.round(mean_absolute_percentage_error(self.y_test, self.y_pred), 3)

        perform = {
            "Metrica": [
                "R2_train", "R2_test",
                "rmse_train", "rmse_test",
                "mae_train", "mae_test",
                "mape_train", "mape_test"
            ],
            "Valor": [
                r2_train, r2_test,
                rmse_train, rmse_test,
                mae_train, mae_test,
                mape_train, mape_test
            ]
        }
        return pd.DataFrame(perform)

In [ ]:
#Instanciamos el modelo
lmodel = Regresion(X_train, X_test, y_train, y_test)

In [ ]:
#Entrenamos y ajustamos
lmodel.fit_reg()

In [ ]:
#Coeficientes del modelo
lmodel.coefs()

In [ ]:
#Validación cruzada
lmodel.cross_values()

In [ ]:
#Performance 
lmodel.performance()

#### Regresión Ridge

In [ ]:
class RidgeRegression:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        
        self.model = None
        self.y_fitted = None
        self.y_pred = None

    #Ajustamos el modelo
    def fit_reg(self, alpha = 1):
        self.model = Ridge(fit_intercept=True, alpha = alpha)
        self.model.fit(self.X_train, self.y_train)
        
        #Calculamos predicciones
        self.y_fitted = self.model.predict(self.X_train)
        self.y_pred = self.model.predict(self.X_test)
        return self.model

    #Coeficientes del modelo
    def coefs(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
            
        cfs = [self.model.intercept_]
        cfs.extend(list(self.model.coef_))
        coef_names = ["intercept"] + list(self.X_train.columns)

        dt = {"Vars": coef_names, "Coefs": cfs}
        df_coefs = pd.DataFrame(dt)
        
        return df_coefs

    #Validacion cruzada
    def cross_values(self, cv=4):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
        
        cv_score = cross_val_score(
            estimator=self.model,
            X=self.X_train,
            y=self.y_train,
            cv=cv,
            n_jobs=-1,
            scoring="r2"
        )
        cv_values = {
            "media": np.round(np.mean(cv_score),3),
            "mediana": np.round(np.median(cv_score),3),
            "desv. est": np.round(np.std(cv_score, ddof=1),3)
        }
        return cv_values

    #Metricas de desempeño
    def performance(self):
        if self.y_fitted is None or self.y_pred is None:
            raise ValueError("Primero ajusta el modelo con fit_reg() para calcular predicciones")
        
        r2_train = np.round(r2_score(self.y_train, self.y_fitted), 3)
        r2_test  = np.round(r2_score(self.y_test, self.y_pred), 3)

        rmse_train = np.round(root_mean_squared_error(self.y_train, self.y_fitted), 3)
        rmse_test  = np.round(root_mean_squared_error(self.y_test, self.y_pred), 3)

        mae_train = np.round(mean_absolute_error(self.y_train, self.y_fitted), 3)
        mae_test  = np.round(mean_absolute_error(self.y_test, self.y_pred), 3)

        mape_train = np.round(mean_absolute_percentage_error(self.y_train, self.y_fitted), 3)
        mape_test  = np.round(mean_absolute_percentage_error(self.y_test, self.y_pred), 3)

        perform = {
            "Metrica": [
                "R2_train", "R2_test",
                "rmse_train", "rmse_test",
                "mae_train", "mae_test",
                "mape_train", "mape_test"
            ],
            "Valor": [
                r2_train, r2_test,
                rmse_train, rmse_test,
                mae_train, mae_test,
                mape_train, mape_test
            ]
        }
        return pd.DataFrame(perform)
    
    #Graficamos las variables penalizadas
    def plot_features(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")

        coefs = pd.Series(self.model.coef_, index=self.X_train.columns)
        coefs_sorted = coefs.sort_values()

        plt.figure(figsize=(10, 6))
        colors = ['#de3163' if c < 0 else 'navy' for c in coefs_sorted]
        coefs_sorted.plot(kind='barh', color=colors)

        plt.title(f'Coeficientes')
        plt.xlabel('Valor del coeficiente (penalizado)')
        plt.ylabel('Variables')
        plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()

#### Búsqueda de hiperparámetros

In [ ]:
#Búsqueda de mejor hiperparametro alpha
ridge_cv = Ridge()

In [ ]:
#Mejor parametro alpha
param_grid = {
    "alpha": uniform(0, 15)
}

In [ ]:
#Búsqueda aleatoria
ridge_search = RandomizedSearchCV(
    estimator=ridge_cv,
    param_distributions=param_grid,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=1989,
    n_jobs=-1
)

In [ ]:
ridge_search.fit(X_train, y_train)

In [ ]:
#Mejores parámetros
best_ridge = ridge_search.best_estimator_
print("Mejor parámetro alpha:", ridge_search.best_params_)

In [ ]:
best_alpha = ridge_search.best_params_["alpha"]

#### Mejor modelo

In [ ]:
#Instanciamos el modelo
ridge_model = RidgeRegression(X_train, X_test, y_train, y_test)

In [ ]:
#Entrenamos y ajustamos
ridge_model.fit_reg(alpha = best_alpha)

In [ ]:
#Coeficientes del modelo
ridge_model.coefs()

In [ ]:
#Validación cruzada
ridge_model.cross_values()

In [ ]:
#Performance 
ridge_model.performance()

In [ ]:
ridge_model.plot_features()

#### Regresión LASSO

In [ ]:
class LassoRegression:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        
        self.model = None
        self.y_fitted = None
        self.y_pred = None

    #Ajustamos el modelo
    def fit_reg(self, alpha = 1):
        self.model = Lasso(fit_intercept=True, alpha = alpha)
        self.model.fit(self.X_train, self.y_train)
        
        #Calculamos predicciones
        self.y_fitted = self.model.predict(self.X_train)
        self.y_pred = self.model.predict(self.X_test)
        return self.model

    #DataFrame con valores observados, ajustados/predichos y residuales
    def fitted_values(self, dataset="train"):
        if dataset == "train":
            df = pd.DataFrame({
                "Observed": self.y_train,
                "Fitted": self.y_fitted,
                "Residual": self.y_train - self.y_fitted
            })
        elif dataset == "test":
            df = pd.DataFrame({
                "Observed": self.y_test,
                "Predicted": self.y_pred,
                "Residual": self.y_test - self.y_pred
            })
        else:
            raise ValueError("dataset debe ser 'train' o 'test'")
        return df

    #Coeficientes del modelo
    def coefs(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
            
        cfs = [self.model.intercept_]
        cfs.extend(list(self.model.coef_))
        coef_names = ["intercept"] + list(self.X_train.columns)

        dt = {"Vars": coef_names, "Coefs": cfs}
        return pd.DataFrame(dt)

    #Validacion cruzada
    def cross_values(self, cv=4):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
        
        cv_score = cross_val_score(
            estimator=self.model,
            X=self.X_train,
            y=self.y_train,
            cv=cv,
            n_jobs=-1,
            scoring="r2"
        )
        cv_values = {
            "media": np.round(np.mean(cv_score),4),
            "mediana": np.round(np.median(cv_score),4),
            "desv. est": np.round(np.std(cv_score, ddof=1),4)
        }
        return cv_values

    #Metricas de desempeño
    def performance(self):
        if self.y_fitted is None or self.y_pred is None:
            raise ValueError("Primero ajusta el modelo con fit_reg() para calcular predicciones")
        
        r2_train = np.round(r2_score(self.y_train, self.y_fitted), 4)
        r2_test  = np.round(r2_score(self.y_test, self.y_pred), 4)

        rmse_train = np.round(root_mean_squared_error(self.y_train, self.y_fitted), 4)
        rmse_test  = np.round(root_mean_squared_error(self.y_test, self.y_pred), 4)

        mae_train = np.round(mean_absolute_error(self.y_train, self.y_fitted), 4)
        mae_test  = np.round(mean_absolute_error(self.y_test, self.y_pred), 4)

        mape_train = np.round(mean_absolute_percentage_error(self.y_train, self.y_fitted), 4)
        mape_test  = np.round(mean_absolute_percentage_error(self.y_test, self.y_pred), 4)

        perform = {
            "Metrica": [
                "R2_train", "R2_test",
                "rmse_train", "rmse_test",
                "mae_train", "mae_test",
                "mape_train", "mape_test"
            ],
            "Valor": [
                r2_train, r2_test,
                rmse_train, rmse_test,
                mae_train, mae_test,
                mape_train, mape_test
            ]
        }
        return pd.DataFrame(perform)
    
    #Graficamos las variables penalizadas
    def plot_features(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")

        coefs = pd.Series(self.model.coef_, index=self.X_train.columns)
        coefs_sorted = coefs.sort_values()

        plt.figure(figsize=(10, 6))
        colors = ['maroon' if c < 0 else 'navy' for c in coefs_sorted]
        coefs_sorted.plot(kind='barh', color=colors)

        plt.title(f'Coeficientes')
        plt.xlabel('Valor del coeficiente (penalizado)')
        plt.ylabel('Variables')
        plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()

#### Búsqueda de hiperparámetro

In [ ]:
#Búsqueda de mejor hiperparametro alpha
lasso_cv = Lasso()

In [ ]:
#Mejor parametro alpha
param_grid = {
    "alpha": uniform(0, 15)
}

In [ ]:
#Búsqueda aleatoria
lasso_search = RandomizedSearchCV(
    estimator=lasso_cv,
    param_distributions=param_grid,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=1989,
    n_jobs=-1
)

In [ ]:
lasso_search.fit(X_train, y_train)

In [ ]:
#Mejores parámetros
best_lasso = lasso_search.best_estimator_
print("Mejor parámetro alpha:", lasso_search.best_params_)

In [ ]:
best_alpha = lasso_search.best_params_["alpha"]

#### Mejor modelo

In [ ]:
#Instanciamos el modelo
lasso_model = LassoRegression(X_train, X_test, y_train, y_test)

In [ ]:
#Entrenamos y ajustamos
lasso_model.fit_reg(alpha = best_alpha)

In [ ]:
#Coeficientes del modelo
lasso_model.coefs()

In [ ]:
#Validación cruzada
lasso_model.cross_values()

In [ ]:
#Performance 
lasso_model.performance()

In [ ]:
#Coeficientes regularizados
lasso_model.plot_features()

#### Elastic Net

In [ ]:
class ElasticNetReg:
    def __init__(self, X_train, X_test, y_train, y_test):
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        
        self.model = None
        self.y_fitted = None
        self.y_pred = None

    #Ajustamos el modelo
    def fit_reg(self, alpha = 1, l1_ratio = 0.5):
        self.model = ElasticNet(fit_intercept=True, alpha = alpha, l1_ratio = l1_ratio)
        self.model.fit(self.X_train, self.y_train)
        
        #Calculamos predicciones
        self.y_fitted = self.model.predict(self.X_train)
        self.y_pred = self.model.predict(self.X_test)
        return self.model

    #Coeficientes del modelo
    def coefs(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
            
        cfs = [self.model.intercept_]
        cfs.extend(list(self.model.coef_))
        coef_names = ["intercept"] + list(self.X_train.columns)

        dt = {"Vars": coef_names, "Coefs": cfs}
        return pd.DataFrame(dt)

    #Validacion cruzada
    def cross_values(self, cv=4):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")
        
        cv_score = cross_val_score(
            estimator=self.model,
            X=self.X_train,
            y=self.y_train,
            cv=cv,
            n_jobs=-1,
            scoring="r2"
        )
        cv_values = {
            "media": np.round(np.mean(cv_score),3),
            "mediana": np.round(np.median(cv_score),3),
            "desv. est": np.round(np.std(cv_score, ddof=1),3)
        }
        return cv_values

    #Metricas de desempeño
    def performance(self):
        if self.y_fitted is None or self.y_pred is None:
            raise ValueError("Primero ajusta el modelo con fit_reg() para calcular predicciones")
        
        r2_train = np.round(r2_score(self.y_train, self.y_fitted), 3)
        r2_test  = np.round(r2_score(self.y_test, self.y_pred), 3)

        rmse_train = np.round(root_mean_squared_error(self.y_train, self.y_fitted), 3)
        rmse_test  = np.round(root_mean_squared_error(self.y_test, self.y_pred), 3)

        mae_train = np.round(mean_absolute_error(self.y_train, self.y_fitted), 3)
        mae_test  = np.round(mean_absolute_error(self.y_test, self.y_pred), 3)

        mape_train = np.round(mean_absolute_percentage_error(self.y_train, self.y_fitted), 3)
        mape_test  = np.round(mean_absolute_percentage_error(self.y_test, self.y_pred), 3)

        perform = {
            "Metrica": [
                "R2_train", "R2_test",
                "rmse_train", "rmse_test",
                "mae_train", "mae_test",
                "mape_train", "mape_test"
            ],
            "Valor": [
                r2_train, r2_test,
                rmse_train, rmse_test,
                mae_train, mae_test,
                mape_train, mape_test
            ]
        }
        return pd.DataFrame(perform)
    
    #Graficamos las variables penalizadas
    def plot_features(self):
        if self.model is None:
            raise ValueError("Primero ajusta el modelo con fit_reg()")

        coefs = pd.Series(self.model.coef_, index=self.X_train.columns)
        coefs_sorted = coefs.sort_values()

        plt.figure(figsize=(10, 6))
        colors = ['#008080' if c < 0 else '#000067' for c in coefs_sorted]
        coefs_sorted.plot(kind='barh', color=colors)

        plt.title(f'Coeficientes')
        plt.xlabel('Valor del coeficiente (penalizado)')
        plt.ylabel('Variables')
        plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
        plt.tight_layout()
        plt.show()

#### Búsqueda de  hiperparámetros

In [ ]:
#Búsqueda de mejor hiperparametro alpha
elastic_cv = ElasticNet()

In [ ]:
#Mejor parametro alpha
param_grid = {
    "alpha": uniform(0, 15),
    "l1_ratio" : uniform(0,1)
}

In [ ]:
#Búsqueda aleatoria
elastic_search = RandomizedSearchCV(
    estimator=elastic_cv,
    param_distributions=param_grid,
    n_iter=20,
    scoring="r2",
    cv=5,
    random_state=1989,
    n_jobs=-1
)

In [ ]:
elastic_search.fit(X_train, y_train)

In [ ]:
#Mejores parámetros
best_elastic = elastic_search.best_estimator_
print("Mejor parámetro alpha:", elastic_search.best_params_)

In [ ]:
best_alpha = elastic_search.best_params_["alpha"]
best_l1 = elastic_search.best_params_["l1_ratio"]

#### Mejor modelo

In [ ]:
#Instanciamos el modelo
elastic_model = ElasticNetReg(X_train, X_test, y_train, y_test)

In [ ]:
#Entrenamos y ajustamos
elastic_model.fit_reg(alpha = best_alpha, l1_ratio = best_l1)

In [ ]:
#Coeficientes del modelo
elastic_model.coefs()

In [ ]:
#Validación cruzada
elastic_model.cross_values()

In [ ]:
#Performance 
elastic_model.performance()

In [ ]:
#Coeficientes regularizados
elastic_model.plot_features()

In [ ]:
all_coefs1 = lmodel.coefs().merge(lasso_model.coefs(), how = "left", on = "Vars", suffixes = ("_OLS", "_LASSO"))
all_coefs2 = ridge_model.coefs().merge(elastic_model.coefs(), how = "left", on = "Vars", suffixes = ("_Ridge", "_ElasticNet"))

In [ ]:
coeficientes = all_coefs1.merge(all_coefs2, how = "left", on = "Vars")

In [ ]:
coeficientes

In [ ]:
pd.concat([lmodel.performance(), lasso_model.performance(), ridge_model.performance(), elastic_model.performance()], axis = 1).drop(["Metrica"], axis = 1)